# 01 · Setup de BigQuery — TodoComponentes

En este notebook se:
1. Autentica contra Google Cloud con la Service Account.
2. Crea el dataset `BQ_DATASET_ID`.
3. Define los esquemas de las 7 tablas con tipos de dato correctos.
4. Crea las tablas **en orden de dependencias FK**.
5. Verifica que todo quedó creado correctamente.

> **Nota:** BigQuery **no aplica restricciones FK físicas** por defecto (básico/editions). Se declaran las `referenced_table` en los esquemas como *documentación* de la relación, y la integridad la garantizan la carga en orden correcto y las queries. El contrato real está en [`docs/er_diagram.dbml`](../docs/er_diagram.dbml) y [`docs/normalizacion.md`](../docs/normalizacion.md).

In [ ]:
!pip install google-cloud-bigquery pandas pandas-gbq dotenv db-dtypes faker

In [ ]:
# --- 1. Autenticación y cliente ---
import os
from pathlib import Path

from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account


# Localizar la raíz del proyecto (donde vive el .env) subiendo desde el cwd
def _find_project_root() -> Path:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / ".env").exists() or (cand / ".env.example").exists():
            return cand
    # Fallback: asumir la estructura conocida del repo
    return Path.cwd().parents[2]

PROJECT_ROOT = _find_project_root()

# Cargar variables de entorno desde el .env de la raíz del proyecto
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET_ID = os.environ["BQ_DATASET_ID"]
CREDENTIALS_PATH = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]

# Resolver ruta relativa de credenciales respecto a la raíz del proyecto
CREDENTIALS_PATH = str((PROJECT_ROOT / CREDENTIALS_PATH).resolve())
assert os.path.exists(CREDENTIALS_PATH), (
    f"Credenciales no encontradas en: {CREDENTIALS_PATH}"
)

# Región en la que viven los datos (EU es el dato por defecto del enunciado)
LOCATION = os.environ.get("BQ_LOCATION", "EU")

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH, scopes=["https://www.googleapis.com/auth/bigquery"]
)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=LOCATION)

print(f"Proyecto : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Región   : {LOCATION}")
print(f"Creds    : {CREDENTIALS_PATH}")

In [ ]:
# --- 2. Crear el dataset (idempotente) ---

dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
client.delete_dataset(dataset_id, delete_contents=True)
dataset = bigquery.Dataset(dataset_id)
dataset.location = LOCATION  # "EU"
dataset.friendly_name = "TodoComponentes — e-commerce de electrónica"
dataset.description = (
    "Modelo relacional 3NF para e-commerce de electrónica. "
    "Team Challenge SQL — Parte II."
)

try:
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset listo: {dataset.project}:{dataset.dataset_id} (loc={dataset.location})")
except Exception as exc:
    print(f"No se pudo crear el dataset: {exc}")
    raise


In [ ]:
# --- 3. Definir esquemas (tipos de dato correctos) ---
# Nota: `bigquery` ya está importado en la celda 2 (from google.cloud import bigquery)
#
# BigQuery NO aplica restricciones FK físicas por defecto. Documentamos la
# relación de cada FK en el campo `description` (no en `referenced_table`, que
# el cliente no expone como kwarg). El contrato real está en docs/er_diagram.dbml
# y docs/normalizacion.md.

schemas: dict[str, list[bigquery.SchemaField]] = {

    "categories": [
        bigquery.SchemaField("category_id", "INT64", description="PK — identificador de categoría"),
        bigquery.SchemaField("name", "STRING", description="Nombre de la categoría"),
        bigquery.SchemaField("description", "STRING", description="Descripción de la categoría"),
    ],

    "customers": [
        bigquery.SchemaField("customer_id", "INT64", description="PK — identificador de cliente"),
        bigquery.SchemaField("first_name", "STRING", description="Nombre"),
        bigquery.SchemaField("last_name", "STRING", description="Apellido"),
        bigquery.SchemaField("email", "STRING", description="Correo electrónico (único)"),
        bigquery.SchemaField("phone", "STRING", description="Teléfono"),
        bigquery.SchemaField("country", "STRING", description="País de residencia"),
        bigquery.SchemaField("city", "STRING", description="Ciudad de residencia"),
        bigquery.SchemaField("acquisition_channel", "STRING", description="Canal de adquisición (organic, paid, referral, ...)"),
        bigquery.SchemaField("created_at", "TIMESTAMP", description="Fecha de alta del cliente"),
    ],

    "products": [
        bigquery.SchemaField("product_id", "INT64", description="PK — identificador de producto"),
        bigquery.SchemaField("category_id", "INT64", description="FK → categories.category_id (1 producto : 1 categoría)"),
        bigquery.SchemaField("name", "STRING", description="Nombre del producto"),
        bigquery.SchemaField("description", "STRING", description="Descripción del producto"),
        bigquery.SchemaField("brand", "STRING", description="Marca"),
        bigquery.SchemaField("unit_cost", "NUMERIC", description="Coste unitario de compra"),
        bigquery.SchemaField("unit_price", "NUMERIC", description="Precio unitario de venta"),
        bigquery.SchemaField("stock", "INT64", description="Unidades en stock"),
        bigquery.SchemaField("is_active", "BOOLEAN", description="¿Está activo en el catálogo?"),
        bigquery.SchemaField("created_at", "TIMESTAMP", description="Fecha de creación del producto"),
    ],

    "orders": [
        bigquery.SchemaField("order_id", "INT64", description="PK — identificador de pedido"),
        bigquery.SchemaField("customer_id", "INT64", description="FK → customers.customer_id (1 cliente : N pedidos)"),
        bigquery.SchemaField("order_status", "STRING", description="Estado (pending, shipped, delivered, cancelled, ...)"),
        bigquery.SchemaField("shipping_country", "STRING", description="País de envío"),
        bigquery.SchemaField("shipping_city", "STRING", description="Ciudad de envío"),
        bigquery.SchemaField("shipping_address", "STRING", description="Dirección completa de envío"),
        bigquery.SchemaField("order_date", "TIMESTAMP", description="Fecha de realización del pedido"),
        bigquery.SchemaField("shipped_date", "TIMESTAMP", description="Fecha de envío (NULL si no ha salido)"),
        bigquery.SchemaField("delivered_date", "TIMESTAMP", description="Fecha de entrega (NULL si no ha llegado)"),
    ],

    "order_items": [
        bigquery.SchemaField("order_item_id", "INT64", description="PK — identificador de línea de pedido"),
        bigquery.SchemaField("order_id", "INT64", description="FK → orders.order_id (1 pedido : N líneas)"),
        bigquery.SchemaField("product_id", "INT64", description="FK → products.product_id (1 producto : N líneas)"),
        bigquery.SchemaField("quantity", "INT64", description="Cantidad de unidades"),
        bigquery.SchemaField("unit_price", "NUMERIC", description="Precio unitario aplicado en la línea (snapshot)"),
        bigquery.SchemaField("discount", "NUMERIC", description="Descuento aplicado en la línea"),
        bigquery.SchemaField("line_total", "NUMERIC", description="Total de la línea (qty * unit_price - discount)"),
    ],

    "payments": [
        bigquery.SchemaField("payment_id", "INT64", description="PK — identificador de pago"),
        bigquery.SchemaField("order_id", "INT64", description="FK → orders.order_id (1 pedido : 1 pago)"),
        bigquery.SchemaField("payment_method", "STRING", description="Método (credit_card, paypal, transfer, ...)"),
        bigquery.SchemaField("payment_status", "STRING", description="Estado (paid, failed, refunded, pending, ...)"),
        bigquery.SchemaField("amount", "NUMERIC", description="Importe pagado"),
        bigquery.SchemaField("paid_at", "TIMESTAMP", description="Fecha y hora del pago"),
    ],

    "reviews": [
        bigquery.SchemaField("review_id", "INT64", description="PK — identificador de reseña"),
        bigquery.SchemaField("order_item_id", "INT64", description="FK → order_items.order_item_id (1 línea : 1 reseña)"),
        bigquery.SchemaField("customer_id", "INT64", description="FK → customers.customer_id (1 cliente : N reseñas)"),
        bigquery.SchemaField("rating", "INT64", description="Valoración 1-5"),
        bigquery.SchemaField("comment", "STRING", description="Comentario de la reseña"),
        bigquery.SchemaField("created_at", "TIMESTAMP", description="Fecha de la reseña"),
    ],
}

# Orden de creación respetando dependencias FK (padre antes que hijo)
CREATE_ORDER = [
    "categories",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews",
]

print("Schemas definidos para:", ", ".join(CREATE_ORDER))

In [ ]:
# --- 4. Crear las tablas en orden de dependencias FK ---

def create_table(table_name: str):
    dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
    table_ref = bigquery.TableReference(dataset_ref, table_name)
    table = bigquery.Table(table_ref)
    table.schema = schemas[table_name]
    table.description = f"TodoComponentes — {table_name}"
    return client.create_table(table, exists_ok=True)

created = []
for name in CREATE_ORDER:
    t = create_table(name)
    created.append(name)
    print(f"✓ {t.table_id}  ({len(t.schema)} columnas)")

print("\nTotal tablas creadas:", len(created))

In [ ]:
# --- 5. Verificación: listar tablas y contadores (deben ser 0 ahora) ---

df = client.query(f"""
    SELECT
    t.table_id,
    t.row_count,
    TIMESTAMP_MILLIS(t.last_modified_time) AS last_modified_ts,
    (t.last_modified_time IS NOT NULL) AS has_timestamp
    FROM `{PROJECT_ID}.{DATASET_ID}.__TABLES__` t
    ORDER BY t.table_id
""").to_dataframe()

# Conteo real de filas por tabla (todas vacías)
counts = {}
for name in CREATE_ORDER:
    counts[name] = client.query(f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET_ID}.{name}`").result().total_rows

print("Tablas y filas:")
for name, n in counts.items():
    print(f"  {name:15s} {n:>8d} filas")

expected = set(CREATE_ORDER)
actual = set(df["table_id"].tolist())
assert expected.issubset(actual), f"Faltan tablas: {expected - actual}"
print("\n✅ Las 7 tablas existen en el dataset.")

---
## Siguiente paso
Pasa al notebook [`02_generate_data.ipynb`](./02_generate_data.ipynb) para poblar el dataset con datos sintéticos (Faker) y validar la carga.